# DSP pipeline y flujo de entrenamiento/inferencia
Este notebook demuestra paso a paso el flujo matemático usado en el proyecto:
preprocesado → filtrado (Butterworth/FIR) → FFT → energía por bandas → energy vector → comparación (coseno + z‑score).
También incluye celdas para preparar conjuntos, augmentaciones, definición de modelo y un esqueleto de entrenamiento/inferencia.

In [ ]:
# 1) Importar librerías comunes
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
from scipy.signal import butter, sosfiltfilt, firwin, lfilter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm

# opcionales (si se dispone de PyTorch)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

print('libs ok, torch available=', TORCH_AVAILABLE)

## 2) Estructura del repositorio y rutas principales
Listaremos rutas relevantes y verificaremos presencia de datos.

In [ ]:
# Ajusta estas rutas según tu workspace
ROOT = Path('..').resolve() if Path('.').name != 'proyecto' else Path('.').resolve()
DATASET_DIR = Path('dataset_aves_processed')
MODELS_DIR = Path('models')
print('root', ROOT)
print('dataset exists', DATASET_DIR.exists())
print('models exists', MODELS_DIR.exists())

## 3) Cargar lista de archivos de audio y etiquetas
Construimos un DataFrame con path y species (si hay subdirectorios por especie).

In [ ]:
def build_file_dataframe(root_dir: Path)->pd.DataFrame:
    rows = []
    if not root_dir.exists():
        return pd.DataFrame(rows, columns=['path','species'])
    for species_dir in sorted(root_dir.iterdir()):
        if not species_dir.is_dir():
            continue
        for f in species_dir.glob('**/*.wav'):
            rows.append({'path': str(f), 'species': species_dir.name})
    return pd.DataFrame(rows)

df = build_file_dataframe(DATASET_DIR)
print('found examples', len(df))
display(df.head())

## 4) Previsualización rápida (waveform y reproducción)
Cargamos un ejemplo y mostramos la forma de onda y un widget de audio (si el entorno lo soporta).

In [ ]:
# seleccionar ejemplo o generar señal de prueba
if len(df) > 0:
    example_path = df.iloc[0]['path']
    y, sr = sf.read(example_path, always_2d=False)
    y = np.mean(y, axis=1) if y.ndim>1 else y
else:
    # generar chirp si no hay archivos
    sr = 22050
    t = np.linspace(0, 2.0, int(2.0*sr), endpoint=False)
    y = 0.5 * np.sin(2*np.pi*500*t) * (1 + 0.5*np.sin(2*np.pi*2*t))

# plot waveform
plt.figure(figsize=(12,3))
librosa.display.waveshow(y, sr=sr)
plt.title('Waveform ejemplo')
plt.show()

# widget de reproducción (si Jupyter lo soporta)
try:
    from IPython.display import Audio, display
    display(Audio(y, rate=sr))
except Exception:
    pass

## 5) Preprocesamiento: mono, remuestreo, trim y normalización

In [ ]:
import librosa.effects as effects

def to_mono(x):
    if x is None or x.size==0:
        return x
    if x.ndim>1:
        return np.mean(x, axis=1).astype(np.float32)
    return x.astype(np.float32)

def resample_audio(y, orig_sr, target_sr):
    if orig_sr == target_sr:
        return y
    return librosa.resample(y, orig_sr, target_sr)

def trim_and_normalize(y, top_db=20):
    y_trim, _ = effects.trim(y, top_db=top_db)
    peak = np.max(np.abs(y_trim)) if y_trim.size>0 else 1.0
    if peak>0:
        y_trim = y_trim / peak
    return y_trim

# ejemplo de uso
y_proc = to_mono(y)
y_proc = resample_audio(y_proc, sr, 22050)
y_proc = trim_and_normalize(y_proc)
plt.figure(figsize=(12,3))
librosa.display.waveshow(y_proc, sr=22050)
plt.title('Waveform preprocesada')
plt.show()

## 6) Extracción de características: STFT, Mel, MFCC y energía por bandas (FFT)
Mostraremos STFT y luego la extracción por bandas usada en el proyecto (FFT magnitud + sumatoria por bandas).

In [ ]:
# STFT y mel
S = np.abs(librosa.stft(y_proc, n_fft=2048, hop_length=512))
S_db = librosa.amplitude_to_db(S, ref=np.max)
plt.figure(figsize=(10,4))
librosa.display.specshow(S_db, sr=22050, hop_length=512, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('STFT (dB)')
plt.show()

# FFT magnitud (rfft) y energía por bandas (ejemplo)
from numpy.fft import rfft, rfftfreq
n = len(y_proc)
if n>0:
    X = rfft(y_proc)
    M = np.abs(X)/n
    freqs = rfftfreq(n, 1/22050)
    plt.figure(figsize=(10,3))
    plt.semilogy(freqs, M+1e-12)
    plt.title('FFT magnitude')
    plt.xlim(0, 11025)
    plt.show()

# ejemplo: dividir [low,high] en bandas y calcular energía
def band_energies_from_fft(mag, freqs, bands):
    energies = []
    for lo, hi in bands:
        mask = (freqs>=lo)&(freqs<hi)
        e = np.sum((mag[mask])**2) if np.any(mask) else 0.0
        energies.append(e)
    return np.array(energies, dtype=np.float32)

bands = np.linspace(1000, 8000, 9)
band_ranges = [(bands[i], bands[i+1]) for i in range(len(bands)-1)]
energies = band_energies_from_fft(M, freqs, band_ranges)
print('band energies', energies)
plt.figure(figsize=(8,3))
plt.bar(range(len(energies)), energies)
plt.title('Energía por banda (ejemplo)')
plt.show()

## 7) Vector de energía (normalización) y firma espectral
Normalizamos las energías para obtener la firma (energy vector) usada por el modelo.

In [ ]:
def energy_vector(energies: np.ndarray)->np.ndarray:
    total = np.sum(energies)
    if total <= 0:
        return np.zeros_like(energies)
    return energies/total

ev = energy_vector(energies)
print('energy vector (sum)', np.sum(ev))
plt.figure(figsize=(8,3))
plt.bar(range(len(ev)), ev)
plt.title('Energy vector (normalized)')
plt.show()

## 8) Ejemplo: construir un modelo simple (profile vector) y calcular similitudes
Mostramos cálculo de similitud coseno y z‑score‑based similarity.

In [ ]:
def cosine_similarity(a,b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na==0 or nb==0:
        return 0.0
    return float(np.dot(a,b)/(na*nb))

def zscore_similarity(obs, mean, std, eps=1e-9):
    safe_std = np.where(std>eps, std, eps)
    z = np.abs(obs-mean)/safe_std
    mean_z = float(np.mean(z))
    return 1.0/(1.0+mean_z)

# perfil toy (por ejemplo tomado promediando ejemplos)
profile = ev + 0.01  # toy profile to avoid zeros
profile = profile/np.sum(profile)
std_profile = np.maximum(0.02*np.ones_like(profile), 1e-6)

c = cosine_similarity(ev, profile)
z = zscore_similarity(ev, profile, std_profile)
score = 0.6*c + 0.4*z
print(f'cosine={c:.4f}, zscore_sim={z:.4f}, combined={score:.4f}')

## 9) (Opcional) Augmentaciones de datos para entrenamiento
Funciones simples: ruido, pitch shift, time stretch, random crop.

In [ ]:
import random
def add_noise(x, snr_db=20):
    rms = np.sqrt(np.mean(x**2))
    noise_rms = rms / (10**(snr_db/20))
    noise = np.random.normal(0, noise_rms, size=x.shape)
    return x + noise

def pitch_shift(x, sr, n_steps):
    return librosa.effects.pitch_shift(x, sr, n_steps)

def time_stretch(x, rate):
    return librosa.effects.time_stretch(x, rate)

def random_crop(x, length):
    if len(x) <= length:
        return x
    start = random.randint(0, len(x)-length)
    return x[start:start+length]

# demo
x_aug = add_noise(y_proc, snr_db=15)
plt.figure(figsize=(10,2))
librosa.display.waveshow(x_aug, sr=22050)
plt.title('Example augmentation (noise)')
plt.show()

## 10) Etiquetas y partición train/val/test (estratificada)

In [ ]:
# construir etiquetas y hacer split estratificado
if len(df)>0:
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['species'])
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(df['path'], df['label']))
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)
    print('train/test', len(df_train), len(df_test))
else:
    print('no dataset available to split')

## 11) Dataset y DataLoader (esqueleto PyTorch)

In [ ]:
if TORCH_AVAILABLE:
    class BirdsongDataset(Dataset):
        def __init__(self, rows, sr=22050, duration=2.0, transform=None):
            self.rows = rows
            self.sr = sr
            self.length = int(sr*duration)
            self.transform = transform

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, idx):
            row = self.rows.iloc[idx]
            y, sr = sf.read(row['path'], always_2d=False)
            y = np.mean(y, axis=1) if y.ndim>1 else y
            y = librosa.resample(y, sr, self.sr) if sr!=self.sr else y
            y = trim_and_normalize(y)
            if len(y) < self.length:
                y = np.pad(y, (0, max(0, self.length-len(y))))
            else:
                y = y[:self.length]
            if self.transform:
                y = self.transform(y)
            # extract features on-the-fly if desired
            return torch.from_numpy(y).float(), int(row['label'])

    print('Dataset skeleton defined')
else:
    print('PyTorch no disponible en este entorno')

## 12) Definición de modelo (CNN simple)

In [ ]:
if TORCH_AVAILABLE:
    class SimpleCNN(nn.Module):
        def __init__(self, input_len, n_classes):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv1d(1,16, kernel_size=9, padding=4),
                nn.ReLU(),
                nn.MaxPool1d(4),
                nn.Conv1d(16,32, kernel_size=9, padding=4),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1)
            )
            self.fc = nn.Linear(32, n_classes)

        def forward(self, x):
            # x: (batch, length) -> add channel
            x = x.unsqueeze(1)
            x = self.conv(x)
            x = x.view(x.size(0), -1)
            return self.fc(x)

    print('SimpleCNN defined')

## 13) Bucle de entrenamiento con checkpoints (esqueleto)

In [ ]:
if TORCH_AVAILABLE:
    def train_one_epoch(model, loader, optimizer, criterion, device='cpu'):
        model.train()
        total_loss = 0.0
        for x,y in loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        return total_loss / len(loader)

    print('training skeleton ready')

## 14) Evaluación y métricas

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate_model(model, loader, device='cpu'):
    model.eval()
    preds = []
    targets = []
    with torch.no_grad() if TORCH_AVAILABLE else nullcontext():
        for x,y in loader:
            x = x.to(device)
            out = model(x)
            p = out.argmax(dim=1).cpu().numpy()
            preds.extend(p.tolist())
            targets.extend(y.numpy().tolist())
    print('acc', accuracy_score(targets, preds))
    print(classification_report(targets, preds))

# Nota: implementar loaders antes de ejecutar

## 15) Inferencia sobre archivo único y exportación de modelo
Ejemplo de cómo cargar un `state_dict` y hacer predicción sobre un archivo.

In [ ]:
def predict_single(model, audio_path, sr=22050, device='cpu'):
    y, orig_sr = sf.read(audio_path, always_2d=False)
    y = np.mean(y, axis=1) if y.ndim>1 else y
    y = librosa.resample(y, orig_sr, sr) if orig_sr!=sr else y
    y = trim_and_normalize(y)
    # adapt to model input length as needed
    if TORCH_AVAILABLE:
        model.eval()
        x = torch.from_numpy(y).unsqueeze(0).float().to(device)
        with torch.no_grad():
            out = model(x)
            probs = torch.softmax(out, dim=1).cpu().numpy()[0]
        return probs
    else:
        return None

## 16) Guardar modelo y artefactos
Guardar `state_dict`, exportar ONNX si es necesario, y guardar CSV con predicciones de muestra.

In [ ]:
if TORCH_AVAILABLE:
    def save_checkpoint(model, path):
        torch.save(model.state_dict(), path)

    def export_onnx(model, example_input, path):
        torch.onnx.export(model, example_input, path, opset_version=12)

    print('save helpers ready')
else:
    print('no torch: skipping save helpers')

---
### Conclusiones y siguientes pasos
- Este notebook muestra el pipeline completo: desde preprocesado hasta esqueleto de entrenamiento.
- Para reproducir los experimentos del proyecto, ajusta rutas, parámetros de extracción y el tamaño de entrada del modelo.